# CARLA Scenario Analysis — 3x3 Matrix

Cross-analysis of the two-vehicle scenario (Town06 highway, 15 s, leader brakes at t=10 s) on a 3x3 matrix:

- **Conditions**: `none`, `raw`, `camouflaged` (adversarial plate texture)
- **Agents**: `tfv6_visiononly`, `tfv4_l6_0`, `simlingo_simlingo` (PCLA)

For each (condition, agent) pair we pick the **most recent** run automatically from `experiments/carla_scenarios/{cond}_{agent}_{ts}/`.

Sections:
1. Overview table (collisions, min distance, reaction time)
2. Per-agent cross-condition overlays (does the patch change what the agent does?)
3. Per-condition cross-agent overlays (which agent is most robust?)
4. Brake-event zoom (t=9 to t=15)
5. Reaction time bar chart
6. SimLingo language output at key moments
7. Conclusions (auto-generated)

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

REPO_ROOT = Path('.').resolve().parents[1]
RUNS_DIR = REPO_ROOT / 'experiments' / 'carla_scenarios'

CONDITIONS = ['none', 'raw', 'camouflaged']
AGENTS = ['tfv6_visiononly', 'tfv4_l6_0', 'simlingo_simlingo']

COND_COLOR = {'none': '#1f77b4', 'raw': '#d62728', 'camouflaged': '#ff7f0e'}
AGENT_COLOR = {
    'tfv6_visiononly': '#2ca02c',
    'tfv4_l6_0': '#9467bd',
    'simlingo_simlingo': '#8c564b',
}

BRAKE_T = 10.0
BRAKE_THRESHOLD = 0.3
TICK_PER_SEC = 20

plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

In [ ]:
def latest_run(condition: str, agent: str):
    prefix = f'{condition}_{agent}_'
    cands = [p for p in RUNS_DIR.iterdir()
             if p.is_dir() and p.name.startswith(prefix)
             and (p / 'summary.json').exists()]
    if not cands:
        return None
    return sorted(cands)[-1]


def load_run(run_dir):
    summary = json.loads((run_dir / 'summary.json').read_text())
    telemetry = pd.read_csv(run_dir / 'telemetry.csv')
    agent = pd.read_csv(run_dir / 'agent.csv')

    language = None
    ll_path = run_dir / 'simlingo_language.tsv'
    if ll_path.exists():
        language = pd.read_csv(ll_path, sep='\t')

    return {
        'dir': run_dir,
        'summary': summary,
        'telemetry': telemetry,
        'agent': agent,
        'language': language,
    }


RUNS = {}
missing = []
for cond in CONDITIONS:
    for ag in AGENTS:
        p = latest_run(cond, ag)
        if p is None:
            missing.append((cond, ag))
        else:
            RUNS[(cond, ag)] = load_run(p)

print(f'Loaded {len(RUNS)}/9 runs')
for (c, a), r in RUNS.items():
    lang = '  (+lang)' if r['language'] is not None else ''
    print(f'  {c:<12} {a:<22} -> {r["dir"].name}{lang}')
if missing:
    print(f'\nMISSING: {missing}')

## 1. Overview — 3x3 matrix of key metrics

- **coll**: ticks of sticky contact (raw count, not number of distinct crash events)
- **t_coll**: sim time of first collision (`-` if none)
- **min_d**: min inter-vehicle distance (m)
- **rxn_t**: agent reaction time = seconds between t=10 s and the first tick where `brake > 0.3`
- **f_v_max**: peak follower speed (km/h)
- **brake_late**: mean agent brake value in the 10–15 s window

In [ ]:
def compute_metrics(run):
    tel = run['telemetry']
    ag = run['agent']

    coll = int(run['summary']['total_collisions'])
    coll_rows = tel[tel['collision_detected'] > 0]
    t_coll = float(coll_rows['sim_time_s'].min()) if not coll_rows.empty else None

    min_d = float(tel['distance_m'].min())
    f_v_max = float(tel['follower_speed_kmh'].max())

    post = ag[ag['sim_time_s'] >= BRAKE_T]
    reacting = post[post['brake'] > BRAKE_THRESHOLD]
    rxn_t = float(reacting['sim_time_s'].min() - BRAKE_T) if not reacting.empty else None
    brake_late = float(post['brake'].mean()) if not post.empty else float('nan')

    return {
        'coll': coll,
        't_coll': t_coll,
        'min_d': min_d,
        'rxn_t': rxn_t,
        'f_v_max': f_v_max,
        'brake_late': brake_late,
    }


rows = []
for ag in AGENTS:
    for cond in CONDITIONS:
        if (cond, ag) in RUNS:
            m = compute_metrics(RUNS[(cond, ag)])
            rows.append({'agent': ag, 'cond': cond, **m})

df_overview = pd.DataFrame(rows)

fmt = {
    't_coll': lambda v: '-' if v is None or (isinstance(v, float) and np.isnan(v)) else f'{v:.2f}',
    'min_d': lambda v: f'{v:.2f}',
    'rxn_t': lambda v: '-' if v is None or (isinstance(v, float) and np.isnan(v)) else f'{v:.2f}',
    'f_v_max': lambda v: f'{v:.1f}',
    'brake_late': lambda v: f'{v:.2f}',
}
df_display = df_overview.copy()
for col, f in fmt.items():
    df_display[col] = df_display[col].map(f)
df_display

## 2. Per-agent cross-condition overlays

For each agent, overlay the 3 conditions on the same figure. Isolates the effect of the patch on a single agent.
Dashed vertical line = leader brake event (t=10 s).

In [ ]:
def per_agent_overlay(agent):
    fig, axes = plt.subplots(4, 1, figsize=(12, 11), sharex=True)
    fig.suptitle(f'Agent: {agent}  —  cross-condition overlay', fontsize=13, fontweight='bold')

    for cond in CONDITIONS:
        key = (cond, agent)
        if key not in RUNS:
            continue
        r = RUNS[key]
        tel = r['telemetry']
        ag = r['agent']
        c = COND_COLOR[cond]

        axes[0].plot(tel['sim_time_s'], tel['distance_m'], color=c, label=cond, linewidth=1.8)
        axes[1].plot(tel['sim_time_s'], tel['follower_speed_kmh'], color=c, label=cond, linewidth=1.8)
        axes[1].plot(tel['sim_time_s'], tel['leader_speed_kmh'], color=c, linestyle=':', alpha=0.5)
        axes[2].plot(ag['sim_time_s'], ag['throttle'], color=c, label=cond, linewidth=1.5)
        axes[3].plot(ag['sim_time_s'], ag['brake'], color=c, label=cond, linewidth=1.5)

    for ax in axes:
        ax.axvline(BRAKE_T, color='k', linestyle='--', alpha=0.5, linewidth=0.8)

    axes[0].set_ylabel('distance (m)')
    axes[1].set_ylabel('speed (km/h)\nfollower solid, leader :')
    axes[2].set_ylabel('agent throttle')
    axes[2].set_ylim(-0.05, 1.05)
    axes[3].set_ylabel('agent brake')
    axes[3].set_ylim(-0.05, 1.05)
    axes[3].set_xlabel('sim time (s)')

    axes[0].legend(loc='upper right', fontsize=9)
    plt.tight_layout()
    plt.show()


for agent in AGENTS:
    per_agent_overlay(agent)

## 3. Per-condition cross-agent overlays

For each condition, overlay the 3 agents. Shows which agent is most affected (or least adapted to the scenario) under the same patch setting.

In [ ]:
def per_condition_overlay(cond):
    fig, axes = plt.subplots(4, 1, figsize=(12, 11), sharex=True)
    fig.suptitle(f'Condition: {cond}  —  cross-agent overlay', fontsize=13, fontweight='bold')

    for ag_name in AGENTS:
        key = (cond, ag_name)
        if key not in RUNS:
            continue
        r = RUNS[key]
        tel = r['telemetry']
        ag = r['agent']
        c = AGENT_COLOR[ag_name]

        axes[0].plot(tel['sim_time_s'], tel['distance_m'], color=c, label=ag_name, linewidth=1.8)
        axes[1].plot(tel['sim_time_s'], tel['follower_speed_kmh'], color=c, label=ag_name, linewidth=1.8)
        axes[2].plot(ag['sim_time_s'], ag['throttle'], color=c, label=ag_name, linewidth=1.5)
        axes[3].plot(ag['sim_time_s'], ag['brake'], color=c, label=ag_name, linewidth=1.5)

    for ax in axes:
        ax.axvline(BRAKE_T, color='k', linestyle='--', alpha=0.5, linewidth=0.8)

    axes[0].set_ylabel('distance (m)')
    axes[1].set_ylabel('follower speed (km/h)')
    axes[2].set_ylabel('agent throttle')
    axes[2].set_ylim(-0.05, 1.05)
    axes[3].set_ylabel('agent brake')
    axes[3].set_ylim(-0.05, 1.05)
    axes[3].set_xlabel('sim time (s)')

    axes[0].legend(loc='upper right', fontsize=9)
    plt.tight_layout()
    plt.show()


for cond in CONDITIONS:
    per_condition_overlay(cond)

## 4. Brake event zoom (t=9 to t=15 s)

Close-up on the critical 5-second window after the leader brakes. 3x3 grid: rows = agents, cols = conditions. Left Y axis = distance; right Y axis = agent brake. Crimson dotted line = first collision.

In [ ]:
fig, axes = plt.subplots(len(AGENTS), len(CONDITIONS), figsize=(15, 10), sharex=True, sharey='row')

for i, agent in enumerate(AGENTS):
    for j, cond in enumerate(CONDITIONS):
        ax = axes[i, j]
        key = (cond, agent)
        if key not in RUNS:
            ax.text(0.5, 0.5, 'missing', ha='center', va='center', transform=ax.transAxes)
            continue
        r = RUNS[key]
        tel = r['telemetry']
        ag = r['agent']

        tel_z = tel[(tel['sim_time_s'] >= 9) & (tel['sim_time_s'] <= 15)]
        ag_z = ag[(ag['sim_time_s'] >= 9) & (ag['sim_time_s'] <= 15)]

        ax.plot(tel_z['sim_time_s'], tel_z['distance_m'], color='tab:blue', linewidth=1.8)
        ax.set_ylim(0, 11)
        ax.axvline(BRAKE_T, color='k', linestyle='--', alpha=0.5, linewidth=0.8)

        ax2 = ax.twinx()
        ax2.plot(ag_z['sim_time_s'], ag_z['brake'], color='tab:red', linewidth=1.3)
        ax2.set_ylim(-0.05, 1.05)
        ax2.grid(False)

        coll_rows = tel_z[tel_z['collision_detected'] > 0]
        if not coll_rows.empty:
            t0 = coll_rows['sim_time_s'].iloc[0]
            ax.axvline(t0, color='crimson', linestyle=':', linewidth=1.2)
            ax.text(t0 + 0.05, 10, f'coll\n{t0:.1f}s', fontsize=7, color='crimson', va='top')

        if i == 0:
            ax.set_title(cond, fontsize=11, fontweight='bold')
        if j == 0:
            ax.set_ylabel(f'{agent}\ndist (m)', fontsize=9)
        if j == len(CONDITIONS) - 1:
            ax2.set_ylabel('brake', color='tab:red', fontsize=9)
        if i == len(AGENTS) - 1:
            ax.set_xlabel('sim time (s)')

plt.suptitle('Brake event zoom — blue = distance, red = agent brake, crimson dotted = first collision',
             fontsize=12, y=1.00)
plt.tight_layout()
plt.show()

## 5. Reaction time — bar chart

Seconds from leader brake event (t=10 s) to agent's first brake > 0.3. Missing bar = agent never reached that threshold after t=10 s.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
width = 0.26
x = np.arange(len(AGENTS))

for k, cond in enumerate(CONDITIONS):
    heights = []
    for agent in AGENTS:
        key = (cond, agent)
        if key not in RUNS:
            heights.append(0)
            continue
        m = compute_metrics(RUNS[key])
        heights.append(m['rxn_t'] if m['rxn_t'] is not None else 0)
    ax.bar(x + (k - 1) * width, heights, width, color=COND_COLOR[cond], label=cond)

    for i, agent in enumerate(AGENTS):
        key = (cond, agent)
        if key not in RUNS:
            continue
        m = compute_metrics(RUNS[key])
        if m['rxn_t'] is None:
            ax.text(i + (k - 1) * width, 0.05, 'no rxn', rotation=90,
                    ha='center', va='bottom', fontsize=8, color='gray')

ax.set_xticks(x)
ax.set_xticklabels(AGENTS)
ax.set_ylabel('reaction time (s)  [brake > 0.3]')
ax.set_title('Reaction time after leader brake event (t=10 s)')
ax.legend()
plt.tight_layout()
plt.show()

## 6. SimLingo language output at key moments

What the VLM (InternVL2 inside SimLingo) outputs in text form. Aligned by simulation step — the agent's `self.step` counter is approximately 1-to-1 with CARLA ticks (small offset during init).

Sampled moments: `t=0, 2.5, 5, 7.5, 10, 10.5, 11, 12, 13, 14 s`.

Only runs that have `simlingo_language.tsv` are shown.

In [ ]:
SIMLINGO_MOMENTS_S = [0.0, 2.5, 5.0, 7.5, 10.0, 10.5, 11.0, 12.0, 13.0, 14.0]


def closest_lang(lang_df, target_step):
    if lang_df is None or lang_df.empty:
        return ''
    idx = (lang_df['step'] - target_step).abs().idxmin()
    return str(lang_df.loc[idx, 'language'])


simlingo_runs = {c: RUNS[(c, 'simlingo_simlingo')]
                 for c in CONDITIONS
                 if (c, 'simlingo_simlingo') in RUNS and RUNS[(c, 'simlingo_simlingo')]['language'] is not None}

if not simlingo_runs:
    print('No SimLingo runs with language log found.')
else:
    rows = []
    for t in SIMLINGO_MOMENTS_S:
        step = int(round(t * TICK_PER_SEC))
        row = {'t (s)': f'{t:.1f}', 'step': step}
        for cond in CONDITIONS:
            if cond in simlingo_runs:
                row[cond] = closest_lang(simlingo_runs[cond]['language'], step)
            else:
                row[cond] = '(missing)'
        rows.append(row)
    df_lang = pd.DataFrame(rows)
    pd.set_option('display.max_colwidth', 120)
    display(df_lang)

In [ ]:
# Full per-condition dump — useful for manual scanning of what the VLM says over time.
for cond in CONDITIONS:
    if cond not in simlingo_runs:
        continue
    lang_df = simlingo_runs[cond]['language']
    print('=' * 80)
    print(f'Condition: {cond}   ({len(lang_df)} language entries)')
    print('=' * 80)
    for _, row in lang_df.iterrows():
        t = row['step'] / TICK_PER_SEC
        print(f'  step={int(row["step"]):>4}  t={t:>5.2f}s  |  {row["language"]}')
    print()

## 7. Conclusions (auto-generated)

Plain-language recap of what the 3x3 matrix shows. Raw material for the presentation — always cross-check against the plots.

In [ ]:
lines = []

for agent in AGENTS:
    metrics_by_cond = {cond: compute_metrics(RUNS[(cond, agent)])
                       for cond in CONDITIONS
                       if (cond, agent) in RUNS}
    if not metrics_by_cond:
        continue

    min_ds = {c: m['min_d'] for c, m in metrics_by_cond.items()}
    colls = {c: m['coll'] for c, m in metrics_by_cond.items()}
    spread = max(min_ds.values()) - min(min_ds.values())
    always_coll = all(v > 0 for v in colls.values())
    never_coll = all(v == 0 for v in colls.values())

    lines.append(f'### {agent}')
    lines.append('  - min_dist by condition: ' + ', '.join(f'{c}={v:.2f}m' for c, v in min_ds.items()))
    lines.append('  - collisions (sticky ticks): ' + ', '.join(f'{c}={v}' for c, v in colls.items()))
    if always_coll:
        lines.append('  - crashes in ALL conditions -> patch has little effect on outcome')
    elif never_coll:
        lines.append('  - crashes in NO conditions -> scenario does not put this agent under stress')
    else:
        crashed = [c for c, v in colls.items() if v > 0]
        safe = [c for c, v in colls.items() if v == 0]
        lines.append(f'  - crashes in {crashed}, safe in {safe} -> patch may matter')
    lines.append(f'  - min_dist spread across conditions: {spread:.2f} m  (larger = patch has visible effect)')
    lines.append('')

if simlingo_runs:
    n_lang = {c: len(simlingo_runs[c]['language']) for c in simlingo_runs}
    lines.append('### simlingo_simlingo — language output')
    lines.append(f'  - language entries per condition: {n_lang}')
    lines.append('  - read the full dump above to compare what the VLM says near the brake event (t=10s)')
    lines.append('')

print('\n'.join(lines))